# AeroCast-NCR — Model Evaluation

Aggregates persisted evaluation artifacts: `models/metrics.json` (from training) and the per-model `predicted_vs_actual_*.csv` files. Reruns `compute_metrics` via `scripts/evaluate_models` to view MAE/RMSE/R² across targets and horizons.

In [ ]:
import json
import os

path = os.path.join("models", "metrics.json")
if os.path.exists(path):
    with open(path) as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2)[:1200])
else:
    print("metrics.json not found — run ml.training.trainer first")

In [ ]:
import glob
import re

import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join("models", "predicted_vs_actual_*.csv"))):
    m = re.match(r"predicted_vs_actual_([a-z0-9_]+)_(\d+)h\.csv", os.path.basename(path))
    if not m:
        continue
    target, horizon = m.group(1), int(m.group(2))
    df = pd.read_csv(path)
    rows.append({
        "target": target,
        "horizon_h": horizon,
        "MAE": (df["actual"] - df["predicted"]).abs().mean(),
        "n_samples": len(df),
    })

summary = pd.DataFrame(rows).pivot(index="target", columns="horizon_h", values="MAE")
summary.columns = [f"MAE_{c}h" for c in summary.columns]
summary.round(2)

In [ ]:
import matplotlib.pyplot as plt

summary.plot(marker="o", figsize=(10, 5), title="MAE by horizon and pollutant")
plt.ylabel("MAE (µg/m³; CO in ppm)")
plt.grid(alpha=0.3)
plt.show()